# 40 - NBIS validation of the new-seed L1 and triplet arms

Run all cells in Colab after notebook 39 completes. After a runtime reset,
run notebook 00 first. Only the two new checkpoints are evaluated.

Notebook 38's frozen validation pairs, corruption cases, six databases,
four conditions (including clean images) and original-input NBIS thresholds
are unchanged. The primary check is triplet vs L1 on `dry_like/severe`.
Report FRR and FAR together, plus paired genuine/impostor transitions.
No test, threshold tuning or automatic winner. Results are saved under
`evaluations/native_triplet_replication_validation_v1`.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import re
import shutil
import subprocess
import sys
import types
import zipfile
import io
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel. A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
REFERENCE_DIR = PROJECT_ROOT / 'evaluations' / 'm1_controlled_validation_v1' / 'bef0c1363b1e6a721afc'
NBIS_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
MINDTCT, BOZORTH3 = (NBIS_ROOT / 'install' / 'bin' / name for name in ('mindtct', 'bozorth3'))
METHODS = ('input', 'm0', 'l1_control', 'l1_sobel')
CONDITIONS = [('identity', 'none')] + [('dry_like', level) for level in ('mild', 'moderate', 'severe')]
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
PRIOR_DIR = PROJECT_ROOT / 'evaluations' / 'alternative_corruption_validation_v1' / '476fc096ee2d51f03921'
REPLICATION_ROOT = PROJECT_ROOT / 'experiments' / 'm3_l1_initialization_v1'
PRIOR_DIR = PROJECT_ROOT / 'evaluations' / 'm2_structure_validation_v1' / '1374b21b3b10b3202a8d'
COMPARISONS = [('input', 'm0'), ('input', 'l1_control'), ('input', 'l1_sobel'), ('l1_control', 'l1_sobel')]

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact checksum mismatch: {directory / name}')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
print('GPU:', torch.cuda.get_device_name(0), '| Evaluate compact dry-specialist diagnostic; no training.')
RISK_ROOT = PROJECT_ROOT / 'experiments' / 'counterfactual_risk_pilot_v1'
PRIOR24_DIR = PROJECT_ROOT / 'evaluations' / 'dry_specialization_validation_v1' / 'a2bcd5551cf4a97e3bf2'

import time
import platform
from pathlib import PurePosixPath
PILOT_RUNS=('native_l1','native_triplet')
METHODS=PILOT_RUNS
PILOT_ROOT=PROJECT_ROOT/'experiments'/'native_triplet_replication_v1'
COMPARISONS=[('native_l1','native_triplet')]


In [ ]:
WORK_DIR=LOCAL_ROOT/'native_pilot_nbis'
REPORT_DIR=WORK_DIR/'reports'
WORK_DIR.mkdir(parents=True,exist_ok=True)
REPORT_DIR.mkdir(parents=True,exist_ok=True)
NBIS_URL = 'https://nigos.nist.gov/nist/nbis/nbis_v5_0_0.zip'
NBIS_SHA256 = '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'
NBIS_BYTES = 52595795
ARCHIVE_DIR = PROJECT_ROOT / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
NBIS_ARCHIVE = ARCHIVE_DIR / 'nbis_v5_0_0.zip'
missing_tools = [name for name in ('curl', 'gcc', 'make', 'sh') if shutil.which(name) is None]
if missing_tools:
    raise RuntimeError(f'Missing build tools: {missing_tools}. Install required tools before retrying.')
if not NBIS_ARCHIVE.exists():
    partial = LOCAL_ROOT / 'nbis_v5_0_0.zip.part'
    if not partial.exists() or partial.stat().st_size != NBIS_BYTES:
        subprocess.run([
            'curl', '--fail', '--location', '--continue-at', '-',
            '--connect-timeout', '30', '--max-time', '600',
            '--output', str(partial), NBIS_URL,
        ], check=True)
    if partial.stat().st_size != NBIS_BYTES or file_digest(partial) != NBIS_SHA256:
        raise RuntimeError('NBIS download integrity failure. Partial file retained; do not build it.')
    pending = NBIS_ARCHIVE.with_suffix('.zip.part')
    shutil.copyfile(partial, pending)
    if file_digest(pending) != NBIS_SHA256:
        raise RuntimeError('NBIS Drive copy checksum mismatch.')
    pending.replace(NBIS_ARCHIVE)
    partial.unlink()
if NBIS_ARCHIVE.stat().st_size != NBIS_BYTES or file_digest(NBIS_ARCHIVE) != NBIS_SHA256:
    raise RuntimeError('Cached NBIS archive differs from the verified release.')
print('Official NBIS source verified:', NBIS_SHA256)
BUILD_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
SOURCE_ROOT = BUILD_ROOT / 'source'
PREFIX = BUILD_ROOT / 'install'
SOURCE = SOURCE_ROOT / 'Rel_5.0.0'
MINDTCT = PREFIX / 'bin' / 'mindtct'
BOZORTH3 = PREFIX / 'bin' / 'bozorth3'
STAMP = BUILD_ROOT / 'verified_build.json'
build_commands = [
    ['sh', 'setup.sh', str(PREFIX), '--without-X11', '--without-OPENJP2', '--64'],
    ['make', 'config'],
    ['make', 'it', 'CC=gcc -fcommon'],
    ['make', 'install', 'LIBNBIS=no', 'CC=gcc -fcommon'],
]
build_signature = {'source_sha256': NBIS_SHA256, 'commands': build_commands}
if STAMP.exists():
    build_record = json.loads(STAMP.read_text(encoding='utf-8'))
    if build_record['signature'] != build_signature:
        raise RuntimeError('Cached NBIS build configuration differs; do not reuse it.')
    for binary in (MINDTCT, BOZORTH3):
        if not binary.is_file() or file_digest(binary) != build_record['binary_sha256'][binary.name]:
            raise RuntimeError(f'Cached NBIS binary changed or is missing: {binary}')
    print('Reusing verified local NBIS build.')
else:
    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
    PREFIX.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(NBIS_ARCHIVE) as archive:
        for member in archive.infolist():
            path = PurePosixPath(member.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in member.filename:
                raise RuntimeError(f'Unsafe source archive path: {member.filename}')
        archive.extractall(SOURCE_ROOT)
    for path in SOURCE.rglob('*'):
        if path.is_file() and (path.name == 'configure' or path.suffix == '.sh'):
            path.chmod(path.stat().st_mode | 0o111)
    build_log = WORK_DIR / 'build.log'
    with build_log.open('w', encoding='utf-8') as output:
        for command in build_commands:
            print('Building:', ' '.join(command), flush=True)
            result = subprocess.run(command, cwd=SOURCE, stdout=output, stderr=subprocess.STDOUT)
            output.flush()
            if result.returncode:
                shutil.copyfile(build_log, REPORT_DIR / 'build.log')
                print('\n'.join(build_log.read_text(errors='replace').splitlines()[-60:]))
                raise RuntimeError(f'NBIS build failed with code {result.returncode}. See saved build.log.')
    shutil.copyfile(build_log, REPORT_DIR / 'build.log')
    build_record = {
        'signature': build_signature,
        'binary_sha256': {binary.name: file_digest(binary) for binary in (MINDTCT, BOZORTH3)},
        'compiler': subprocess.run(['gcc', '--version'], check=True, capture_output=True, text=True).stdout,
        'platform': platform.platform(),
    }
    STAMP.write_text(json.dumps(build_record, indent=2), encoding='utf-8')
versions = {}
for binary in (MINDTCT, BOZORTH3):
    result = subprocess.run([str(binary), '-version'], check=True, capture_output=True, text=True)
    versions[binary.name] = result.stdout + result.stderr
    print(binary.name, versions[binary.name].strip())
(REPORT_DIR / 'build_provenance.json').write_text(
    json.dumps({**build_record, 'versions': versions}, indent=2), encoding='utf-8')

In [ ]:

pilot_pointer=json.loads((PILOT_ROOT/'completed_experiment.json').read_text(encoding='utf-8'))
PILOT_DIR=Path(pilot_pointer['directory'])
if (PILOT_DIR.parent!=PILOT_ROOT or PILOT_DIR.name!=pilot_pointer['configuration_sha256'][:20]
        or file_digest(PILOT_DIR/'configuration.json')!=pilot_pointer['configuration_sha256']
        or file_digest(PILOT_DIR/'completion.json')!=pilot_pointer['completion_sha256']):
    raise RuntimeError('Completed37 pointer/configuration/completion differs.')
pilot_config=json.loads((PILOT_DIR/'configuration.json').read_text(encoding='utf-8'))
pilot_completion=json.loads((PILOT_DIR/'completion.json').read_text(encoding='utf-8'))
if (pilot_config['version']!='native_triplet_replication_v1'
        or pilot_completion['status']!='native_counterfactual_pilot_complete'
        or pilot_completion['configuration_sha256']!=pilot_pointer['configuration_sha256']
        or pilot_config['run_names']!=list(PILOT_RUNS) or pilot_completion['run_names']!=list(PILOT_RUNS)
        or pilot_config['epochs']!=2 or pilot_completion['epochs']!=2
        or pilot_config['effective_batch']!=8 or pilot_config['seed']!=20261127
        or pilot_config['frozen_selector_seed']!=20261110):
    raise RuntimeError('Expected the completed two-arm, two-epoch seed replication from notebook 39.')
verify_files(PILOT_DIR,pilot_completion['file_sha256'])
split_protocol=json.loads((SPLIT_DIR/'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR,split_protocol['file_sha256'])
locked=pd.read_csv(SPLIT_DIR/'manifest_splits.csv',keep_default_na=False)
validation=locked[locked['split'].eq('validation')].sort_values('relative_path').reset_index(drop=True)
pair_config=json.loads((PAIR_DIR/'configuration.json').read_text(encoding='utf-8'))
sim_config=json.loads((SIM_DIR/'configuration.json').read_text(encoding='utf-8'))
m0_config=json.loads((M0_DIR/'configuration.json').read_text(encoding='utf-8'))
verify_files(M0_DIR,m0_config['source_file_sha256'])
for key,path in [('pair_configuration_sha256',PAIR_DIR/'configuration.json'),
        ('simulator_configuration_sha256',SIM_DIR/'configuration.json'),
        ('validation_sources_sha256',M0_DIR/'validation_sources.csv')]:
    if pilot_config[key]!=file_digest(path):
        raise RuntimeError(f'37/38 protocol differs: {key}')
for config in (pair_config,sim_config,m0_config):
    if config['split_protocol_sha256']!=file_digest(SPLIT_DIR/'protocol.json'):
        raise RuntimeError('Split provenance differs.')
if (file_digest(PAIR_DIR/'pairs.csv')!=pair_config['pair_csv_sha256']
        or file_digest(SIM_DIR/'fingerprint_degradation.py')!=sim_config['module_sha256']
        or build_record['signature']['source_sha256']!=pair_config['nbis_source_sha256']
        or pair_config['mindtct_flags']!=[] or pair_config['bozorth3_flags']!=['-A','minminutiae=10']):
    raise RuntimeError('Frozen simulator/NBIS/pair parameters differ.')
for key,value in [('torch_version',str(torch.__version__)),('torch_cuda',str(torch.version.cuda)),
        ('numpy_version',np.__version__),('pillow_version',Image.__version__)]:
    if pilot_config[key]!=value or m0_config[key]!=value:
        raise RuntimeError(f'Frozen training/evaluation library differs: {key}')
if (sim_config['numpy_version']!=np.__version__ or sim_config['pillow_version']!=Image.__version__
        or pair_config['pillow_version']!=Image.__version__):
    raise RuntimeError('Normalization/simulator version differs.')
pairs=pd.read_csv(PAIR_DIR/'pairs.csv',keep_default_na=False)
probe_records=validation[validation['impression_id'].ne(1)].to_dict('records')
gallery_records=validation[validation['impression_id'].eq(1)].to_dict('records')
if (validation.groupby(['dataset','db']).ngroups!=6 or set(validation.dataset)!={'FVC2000','FVC2002'}
        or pairs.pair_id.duplicated().any() or pairs.duplicate_sensitivity_eligible.dtype!=bool
        or set(pairs.probe_path)!={r['relative_path'] for r in probe_records}
        or set(pairs.gallery_path)!={r['relative_path'] for r in gallery_records}):
    raise RuntimeError('Expected complete frozen development-validation pairs; no tests allowed.')
for record in validation.to_dict('records'):
    path=DATA_ROOT/record['relative_path']
    if not path.is_file() or file_digest(path)!=record['sha256']:
        raise RuntimeError(f'Validation bytes missing/changed: {path}. Run00; do not alter splits.')
training=load_module('native_pilot_nb38_model',M0_DIR/'restoration_baseline.py')
simulator=load_module('native_pilot_nb38_simulator',SIM_DIR/'fingerprint_degradation.py')
reference_config=json.loads((REFERENCE_DIR/'configuration.json').read_text(encoding='utf-8'))
reference_completion=json.loads((REFERENCE_DIR/'completion.json').read_text(encoding='utf-8'))
verify_files(REFERENCE_DIR,reference_completion['file_sha256'])
helper_path=REFERENCE_DIR/'frozen_nb08_helpers.py.txt'
if file_digest(helper_path)!=reference_config['frozen_helpers_sha256']:
    raise RuntimeError('Established NBIS helper checksum differs.')
HELPER_SOURCE=helper_path.read_text(encoding='utf-8')
prior_config=json.loads((PRIOR_DIR/'configuration.json').read_text(encoding='utf-8'))
prior_completion=json.loads((PRIOR_DIR/'completion.json').read_text(encoding='utf-8'))
verify_files(PRIOR_DIR,prior_completion['file_sha256'])
if (prior_completion['status']!='structure_validation_matching_complete'
        or prior_completion['configuration_sha256']!=file_digest(PRIOR_DIR/'configuration.json')
        or prior_config['pair_configuration_sha256']!=file_digest(PAIR_DIR/'configuration.json')
        or prior_config['pair_csv_sha256']!=file_digest(PAIR_DIR/'pairs.csv')
        or prior_config['simulator_configuration_sha256']!=file_digest(SIM_DIR/'configuration.json')
        or prior_config['frozen_helpers_sha256']!=file_digest(helper_path)
        or prior_config['inference_and_matcher']['nbis_binary_sha256']!=build_record['binary_sha256']):
    raise RuntimeError('Saved original NBIS thresholds refer to a different protocol/binary build.')
threshold_table=pd.read_csv(PRIOR_DIR/'validation_thresholds.csv')
base=threshold_table[threshold_table.condition.eq('identity/none') & threshold_table.method.eq('input')]
if base.duplicated(['dataset','db']).any():
    raise RuntimeError('Duplicate saved input thresholds.')
fixed_thresholds={(r['dataset'],r['db']):r['original_fixed_threshold'] for r in base.to_dict('records')}
if (set(fixed_thresholds)!=set(zip(validation.dataset,validation.db))
        or any(not math.isfinite(v) or v<0 or v!=int(v) for v in fixed_thresholds.values())):
    raise RuntimeError('Incomplete/noninteger saved per-DB input thresholds.')
print('Frozen NBIS pairs:',len(pairs),'| probes:',len(probe_records),'| methods:',METHODS)


In [ ]:

def load_selected(directory,pointer,expected_sha,run):
    name=pointer['file']
    if (Path(name).name!=name or '/' in name or '\\' in name or name in ('.','..')
            or pointer['configuration_sha256']!=expected_sha
            or file_digest(directory/name)!=pointer['sha256']):
        raise RuntimeError('Selected model pointer/integrity differs.')
    state=torch.load(directory/name,map_location='cpu',weights_only=True)
    if (state['configuration_sha256']!=expected_sha or state['run']!=run
            or state['completed_epochs']!=pointer['completed_epochs']):
        raise RuntimeError('Selected model payload/epoch differs.')
    with torch.random.fork_rng(devices=[]):
        model=training.RestorationUNet()
    model.load_state_dict(state['model'],strict=True)
    if not all(torch.isfinite(value).all() for value in model.state_dict().values()):
        raise RuntimeError('Nonfinite selected weights.')
    return model.to(DEVICE).eval()

original_pointer=json.loads((SPECIALIZATION_ROOT/'completed_experiment.json').read_text(encoding='utf-8'))
ORIGINAL_DIR=Path(original_pointer['directory'])
if (ORIGINAL_DIR.parent!=SPECIALIZATION_ROOT
        or file_digest(ORIGINAL_DIR/'configuration.json')!=original_pointer['configuration_sha256']
        or file_digest(ORIGINAL_DIR/'completion.json')!=original_pointer['completion_sha256']
        or original_pointer['configuration_sha256']!=pilot_config['original23_configuration_sha256']):
    raise RuntimeError('Original23 warmstart provenance differs.')
original_completion=json.loads((ORIGINAL_DIR/'completion.json').read_text(encoding='utf-8'))
original_run=ORIGINAL_DIR/'dry_l1_seed20261110'
if file_digest(original_run/'completion.json')!=original_completion['run_completion_sha256']['dry_l1_seed20261110']:
    raise RuntimeError('Original23 completed run differs.')
original_marker=json.loads((original_run/'completion.json').read_text(encoding='utf-8'))
verify_files(original_run,original_marker['file_sha256'])
original_best=json.loads((original_run/'best.json').read_text(encoding='utf-8'))
if original_best!=pilot_config['original23_best'] or original_best!=original_marker['result']['best_checkpoint']:
    raise RuntimeError('37/38 original23 checkpoint differs.')
models={}
model_pointers={}
for name in PILOT_RUNS:
    directory=PILOT_DIR/name
    if file_digest(directory/'completion.json')!=pilot_completion['run_completion_sha256'][name]:
        raise RuntimeError('Pilot arm completion hash differs.')
    marker=json.loads((directory/'completion.json').read_text(encoding='utf-8'))
    verify_files(directory,marker['file_sha256'])
    selected=json.loads((directory/'best.json').read_text(encoding='utf-8'))
    if (marker['configuration_sha256']!=pilot_pointer['configuration_sha256'] or marker['run']!=name
            or marker['epochs_completed']!=2 or marker['best_checkpoint']!=selected
            or not 0<=selected['completed_epochs']<=2):
        raise RuntimeError('Pilot completed budget/selection differs.')
    models[name]=load_selected(directory,selected,pilot_pointer['configuration_sha256'],name)
    model_pointers[name]=selected
print('Verified both COMPLETED replication arms; selected epochs:',
    {name:value['completed_epochs'] for name,value in model_pointers.items()})


In [ ]:

evaluation_config=dict(version='native_triplet_replication_validation_v1',implementation_sha256='bb1b3985113c595b3ef5ed33dc17caf4ac4a180ea8f6e26c9ab9d2b0cf3c9e75',
    pilot_configuration_sha256=pilot_pointer['configuration_sha256'],
    pilot_completion_sha256=pilot_pointer['completion_sha256'],
    pair_configuration_sha256=file_digest(PAIR_DIR/'configuration.json'),
    pair_csv_sha256=file_digest(PAIR_DIR/'pairs.csv'),split_protocol_sha256=file_digest(SPLIT_DIR/'protocol.json'),
    simulator_configuration_sha256=file_digest(SIM_DIR/'configuration.json'),
    frozen_helpers_sha256=file_digest(helper_path),nbis_binary_sha256=build_record['binary_sha256'],
    thresholds_sha256=file_digest(PRIOR_DIR/'validation_thresholds.csv'),
    model_checkpoints=model_pointers,methods=list(METHODS),conditions=CONDITIONS,
    comparisons=COMPARISONS,threshold_policy='saved14 original-input threshold; no recalibration',
    inference='one full native normalized image, FP32, no AMP, no padding, no tiles',
    quantization='numpy.rint(image*255).astype(uint8); PNG500dpi',independent_test=False,
    torch_version=str(torch.__version__),torch_cuda=str(torch.version.cuda),
    numpy_version=np.__version__,pillow_version=Image.__version__,
    gpu_name=torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None)
evaluation_sha=hashlib.sha256(json_bytes(evaluation_config)).hexdigest()
EVALUATION_ROOT=PROJECT_ROOT/'evaluations'/'native_triplet_replication_validation_v1'
REPORT_DIR=EVALUATION_ROOT/evaluation_sha[:20]
CACHE_DIR=LOCAL_ROOT/'native_counterfactual_validation'/evaluation_sha[:20]
TEMPLATE_DIR=CACHE_DIR/'templates'
for directory in (REPORT_DIR,CACHE_DIR,TEMPLATE_DIR):
    directory.mkdir(parents=True,exist_ok=True)
for name,data in {'configuration.json':json_bytes(evaluation_config),'frozen_helpers.py.txt':HELPER_SOURCE.encode()}.items():
    path=REPORT_DIR/name
    if path.exists() and path.read_bytes()!=data:
        raise RuntimeError('Frozen native NBIS evaluation changed.')
    if not path.exists():
        write_verified(path,data)
active_helpers=types.ModuleType('native_nb38_helpers')
active_helpers.__dict__.update(globals())
exec(compile(HELPER_SOURCE,'<unchanged-nbis-helpers>','exec'),active_helpers.__dict__)
validation_paths=frozenset(validation.relative_path)
def condition_input38(record,family,severity):
    if record['relative_path'] not in validation_paths or (family,severity) not in CONDITIONS:
        raise RuntimeError('Evaluation may open only frozen development-validation images/conditions.')
    dpi=pair_config['source_dpi'][f"{record['dataset']}_{record['db']}"]
    reference=training.normalized_image(str(DATA_ROOT/record['relative_path']),dpi)
    if family=='identity':
        return reference,reference.copy()
    degraded,_=simulator.degrade(reference,family,severity,
        simulator.case_seed(record['relative_path'],family,severity,sim_config['seed']))
    return reference,degraded
active_helpers.condition_input=condition_input38
units38={}


In [ ]:

pending=[(family,severity,method) for family,severity in CONDITIONS for method in METHODS
    if active_helpers.load_unit(REPORT_DIR/f'{family}_{severity}_{method}') is None]
gallery_frame=None
if pending:
    gallery_rows=[]
    for index,record in enumerate(gallery_records,1):
        reference,_=condition_input38(record,'identity','none')
        gallery_rows.append({**record,**active_helpers.extract_png(active_helpers.quantize(reference))})
        if index%50==0 or index==len(gallery_records):
            print(f'Original galleries {index}/{len(gallery_records)}',flush=True)
    gallery_frame=pd.DataFrame(gallery_rows)
    write_csv(REPORT_DIR/'gallery_extraction.csv',gallery_frame)
for family,severity in CONDITIONS:
    for method in METHODS:
        unit=f'{family}_{severity}_{method}'
        directory=REPORT_DIR/unit
        loaded=active_helpers.load_unit(directory)
        if loaded is not None:
            units38[unit]=loaded
            print('Verified/resumed NBIS variant:',unit,flush=True)
            continue
        directory.mkdir(parents=True,exist_ok=True)
        work=CACHE_DIR/unit
        work.mkdir(parents=True,exist_ok=True)
        if method!='input':
            active_helpers.model=models[method]
        rows=[]
        started38=time.monotonic()
        for index,record in enumerate(probe_records,1):
            reference,degraded=condition_input38(record,family,severity)
            output=degraded if method=='input' else active_helpers.restore_image(degraded)
            rows.append({**record,**active_helpers.extract_png(active_helpers.quantize(output)),
                **active_helpers.pixel_metrics(reference,degraded,output),
                'condition':f'{family}/{severity}','method':method})
            if index%100==0 or index==len(probe_records):
                print(f'{unit}: images {index}/{len(probe_records)}, {(time.monotonic()-started38)/60:.1f}min',flush=True)
        extraction=pd.DataFrame(rows)
        scored=active_helpers.match_pairs(extraction,gallery_frame,work)
        write_csv(directory/'scores.csv',scored)
        write_csv(directory/'extraction.csv',extraction)
        active_helpers.archive_templates(directory/'templates_and_logs.zip',rows+gallery_frame.to_dict('records'),work)
        marker=dict(configuration_sha256=evaluation_sha,variant=unit,
            file_sha256={name:file_digest(directory/name) for name in
                ('scores.csv','extraction.csv','templates_and_logs.zip')})
        write_verified(directory/'completion.json',json_bytes(marker))
        units38[unit]=active_helpers.load_unit(directory)
        print('Saved NBIS variant:',unit,flush=True)


In [ ]:
METRIC_SOURCE = "def paired_attempt_changes(before,after,thresholds,condition,baseline,method):\n    columns=['pair_id','dataset','db','label','matchable','score']\n    joined=before[columns].merge(after[columns],on='pair_id',suffixes=('_before','_after'),\n        how='outer',validate='one_to_one',indicator=True)\n    if not joined['_merge'].eq('both').all() or any(\n            not joined[f'{key}_before'].equals(joined[f'{key}_after']) for key in ('dataset','db','label')):\n        raise RuntimeError('Paired NBIS metadata/attempts differ.')\n    rows=[]\n    for (dataset,db),group in joined.groupby(['dataset_before','db_before'],sort=True):\n        threshold=thresholds[(dataset,db)]\n        accepted_before=group.matchable_before & group.score_before.ge(threshold)\n        accepted_after=group.matchable_after & group.score_after.ge(threshold)\n        genuine=group.label_before.eq(1)\n        impostor=~genuine\n        rows.append(dict(condition=condition,baseline=baseline,method=method,dataset=dataset,db=db,\n            genuine_attempts=int(genuine.sum()),impostor_attempts=int(impostor.sum()),\n            genuine_recoveries=int((genuine & ~accepted_before & accepted_after).sum()),\n            genuine_harms=int((genuine & accepted_before & ~accepted_after).sum()),\n            newly_accepted_impostors=int((impostor & ~accepted_before & accepted_after).sum()),\n            newly_rejected_impostors=int((impostor & accepted_before & ~accepted_after).sum())))\n    return rows\n"
exec(compile(METRIC_SOURCE,'<paired-all-attempts>','exec'))


In [ ]:

metric_rows,pixel_rows,paired_rows38=[],[],[]
for family,severity in CONDITIONS:
    condition=f'{family}/{severity}'
    for method in METHODS:
        scored,extraction=units38[f'{family}_{severity}_{method}']
        for (dataset,db),frame in scored.groupby(['dataset','db'],sort=True):
            for view,selected in [('all_impressions',frame),
                    ('byte_distinct_sensitivity',frame[frame.duplicate_sensitivity_eligible])]:
                values=active_helpers.metrics_at_threshold(selected,fixed_thresholds[(dataset,db)])
                metric_rows.append(dict(condition=condition,method=method,dataset=dataset,db=db,view=view,**values))
        for (dataset,db),frame in extraction.groupby(['dataset','db'],sort=True):
            pixel_rows.append(dict(condition=condition,method=method,dataset=dataset,db=db,
                images=len(frame),image_failure_rate=float(frame.status.ne('ok').mean()),
                float_output_mae=float(frame.float_output_mae.mean()),
                png_output_mae=float(frame.png_output_mae.mean())))
    for baseline,method in COMPARISONS:
        paired_rows38.extend(paired_attempt_changes(
            units38[f'{family}_{severity}_{baseline}'][0],units38[f'{family}_{severity}_{method}'][0],
            fixed_thresholds,condition,baseline,method))
metrics38=pd.DataFrame(metric_rows)
macro38=metrics38[metrics38['view'].eq('all_impressions')].groupby(['condition','method'],sort=True).agg(
    databases=('db','size'),attempt_frr=('attempt_frr','mean'),attempt_far=('attempt_far','mean')).reset_index()
if not macro38.databases.eq(6).all():
    raise RuntimeError('No partial-database macro averages allowed.')
dry38=macro38[macro38.condition.str.startswith('dry_like/')].groupby('method',sort=True).agg(
    conditions=('condition','size'),attempt_frr=('attempt_frr','mean'),attempt_far=('attempt_far','mean')).reset_index()
if len(dry38)!=len(METHODS) or not dry38.conditions.eq(3).all():
    raise RuntimeError('Incomplete dry-condition/method coverage.')
comparisons38=[]
for scope,frame in [('dry_macro',dry38)]+list(macro38.groupby('condition',sort=True)):
    points=frame.set_index('method')
    for baseline,method in COMPARISONS:
        left,right=points.loc[baseline],points.loc[method]
        comparisons38.append(dict(scope=scope,baseline=baseline,method=method,
            delta_frr_pp=100*float(right.attempt_frr-left.attempt_frr),
            delta_far_pp=100*float(right.attempt_far-left.attempt_far)))
for name,frame in [('validation_metrics.csv',metrics38),('macro_metrics.csv',macro38),
        ('dry_aggregate.csv',dry38),('paired_attempt_changes.csv',pd.DataFrame(paired_rows38)),
        ('pixel_and_failures.csv',pd.DataFrame(pixel_rows)),
        ('operating_point_comparisons.csv',pd.DataFrame(comparisons38))]:
    write_csv(REPORT_DIR/name,frame)
print('Equal DB/condition weights, FIXED saved thresholds. Lower FRR with higher FAR is a tradeoff.')
print(dry38.to_string(index=False))
print(pd.DataFrame(comparisons38).to_string(index=False))


In [ ]:

verify_files(PILOT_DIR,pilot_completion['file_sha256'])
if file_digest(PILOT_DIR/'completion.json')!=pilot_pointer['completion_sha256']:
    raise RuntimeError('Notebook 39 completion changed during validation.')
summary38=dict(status='native_counterfactual_validation_complete',configuration_sha256=evaluation_sha,
    pilot_configuration_sha256=pilot_pointer['configuration_sha256'],
    training_seed=pilot_config['seed'],methods=list(METHODS),
    variants=len(METHODS)*len(CONDITIONS),independent_test=False,training_performed=False,
    external_test_evaluated=False,
    unit_completion_sha256={key:file_digest(REPORT_DIR/key/'completion.json') for key in units38},
    file_sha256={name:file_digest(REPORT_DIR/name) for name in ('configuration.json','frozen_helpers.py.txt',
        'gallery_extraction.csv','validation_metrics.csv','macro_metrics.csv','dry_aggregate.csv',
        'paired_attempt_changes.csv','pixel_and_failures.csv','operating_point_comparisons.csv')})
write_verified(REPORT_DIR/'completion.json',json_bytes(summary38))
write_verified(EVALUATION_ROOT/'completed_evaluation.json',json_bytes(dict(directory=str(REPORT_DIR),
    configuration_sha256=evaluation_sha,completion_sha256=file_digest(REPORT_DIR/'completion.json'))))
print(json.dumps(summary38,indent=2))
print('NBIS40 REPLICATION COMPLETE:',REPORT_DIR)
primary=next(row for row in comparisons38 if row['scope']=='dry_like/severe')
print(f"Severe | triplet vs L1: FRR {primary['delta_frr_pp']:+.3f} pp; FAR {primary['delta_far_pp']:+.3f} pp")
print('Also inspect mild, moderate and clean-image results above. No automatic winner or test claim.')
print('Send back dry_aggregate.csv, operating_point_comparisons.csv and paired_attempt_changes.csv.')
